In [3]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class BertrandPricingEnv(gym.Env):
    """
    Two-firm simultaneous price-setting environment.

    Demand for firm i (linear demand Q = a - bP, split on ties):
        Q_i = a - b*p_i        if p_i <  p_j   (firm i undercuts -> full demand)
        Q_i = 0.5*(a - b*p_i)  if p_i == p_j   (tie -> split demand)
        Q_i = 0                if p_i >  p_j   (firm i priced out)

    Profit for firm i:
        pi_i = (p_i - marginal_cost) * Q_i

    Action space (per firm): Discrete(n_price_levels)
        action index -> price via a linear grid from 0 to max_price
    Observation: last round's two prices, normalized to [0, 1]
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, a=100.0, b=1.0, marginal_cost=10.0,
                 n_price_levels=25, max_price=None, max_steps=20):
        super().__init__()

        # --- configurable market parameters ---
        self.a = a                          # demand intercept
        self.b = b                          # demand slope
        self.marginal_cost = marginal_cost  # c
        self.n_price_levels = n_price_levels
        # if no ceiling given, default to the monopoly-ish price a/b so the
        # grid covers the economically meaningful range
        self.max_price = max_price if max_price is not None else (a / b)
        self.max_steps = max_steps

        # price grid: index -> actual price
        self.price_grid = np.linspace(0.0, self.max_price, self.n_price_levels)

        # --- Gymnasium spaces ---
        # two firms, each picks one discrete price level
        self.action_space = spaces.MultiDiscrete([n_price_levels, n_price_levels])

        # observation: last prices for both firms, normalized to [0,1]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(2,), dtype=np.float32
        )

        self._step_count = 0
        self._last_prices = np.zeros(2, dtype=np.float32)

    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._step_count = 0
        # start both firms at price 0 (arbitrary, agents will overwrite immediately)
        self._last_prices = np.zeros(2, dtype=np.float32)
        observation = self._get_obs()
        info = {"prices": self._last_prices.copy()}
        return observation, info

    # ------------------------------------------------------------------
    def step(self, actions):
        """
        actions: array-like of two ints, each in [0, n_price_levels-1]
        """
        actions = np.asarray(actions)
        p1 = self.price_grid[actions[0]]
        p2 = self.price_grid[actions[1]]
        prices = np.array([p1, p2], dtype=np.float32)

        q1, q2 = self._demand(p1, p2)
        profit1 = (p1 - self.marginal_cost) * q1
        profit2 = (p2 - self.marginal_cost) * q2

        self._last_prices = prices
        self._step_count += 1

        observation = self._get_obs()
        reward = np.array([profit1, profit2], dtype=np.float32)

        terminated = False  # no natural "win/lose" end state
        truncated = self._step_count >= self.max_steps

        info = {
            "prices": prices,
            "quantities": np.array([q1, q2], dtype=np.float32),
            "profits": reward,
        }
        return observation, reward, terminated, truncated, info

    # ------------------------------------------------------------------
    def _demand(self, p1, p2):
        """Piecewise Bertrand demand split. Returns (q1, q2)."""
        def q_if_alone(p):
            return max(self.a - self.b * p, 0.0)

        if p1 < p2:
            return q_if_alone(p1), 0.0
        elif p2 < p1:
            return 0.0, q_if_alone(p2)
        else:  # tie -> split
            q = q_if_alone(p1)
            return 0.5 * q, 0.5 * q

    # ------------------------------------------------------------------
    def _get_obs(self):
        # normalize prices into [0,1] using max_price
        return (self._last_prices / self.max_price).astype(np.float32)

    # ------------------------------------------------------------------
    def render(self):
        p1, p2 = self._last_prices
        print(f"step={self._step_count}  p1={p1:.2f}  p2={p2:.2f}")

    # ------------------------------------------------------------------
    def price_index_for(self, price):
        """Helper: find the closest action index for a target price.
        Useful for unit testing against the analytical Nash price."""
        return int(np.argmin(np.abs(self.price_grid - price)))


# ======================================================================
# UNIT TESTS — analytical Nash price check
# ======================================================================
def run_unit_tests():
    # Modified n_price_levels to ensure marginal_cost is an exact point on the price grid
    env = BertrandPricingEnv(a=100, b=1, marginal_cost=10, n_price_levels=11, max_price=100)
    env.reset()

    c = env.marginal_cost
    c_idx = env.price_index_for(c)

    print("=== TEST 1: both firms price at marginal cost (Bertrand NE) ===")
    obs, reward, term, trunc, info = env.step([c_idx, c_idx])
    print(f"prices={info['prices']}  profits={info['profits']}")
    assert np.allclose(info["profits"], [0, 0], atol=1.0), \
        "FAIL: profits should be ~0 when both firms price at marginal cost"
    print("PASS: both profits ~0 at p1=p2=c\n")

    print("=== TEST 2: firm 1 undercuts slightly below firm 2 ===")
    above_c_idx = env.price_index_for(c + 20)  # firm 2 prices above cost
    env.reset()
    obs, reward, term, trunc, info = env.step([c_idx, above_c_idx])
    print(f"prices={info['prices']}  quantities={info['quantities']}  profits={info['profits']}")
    assert info["quantities"][1] == 0.0, \
        "FAIL: higher-priced firm should get zero demand"
    assert info["quantities"][0] > 0.0, \
        "FAIL: lower-priced firm should capture all demand"
    print("PASS: lower price captures full market, higher price gets zero\n")

    print("=== TEST 3: tie -> demand splits evenly ===")
    tie_idx = env.price_index_for(c + 20)
    env.reset()
    obs, reward, term, trunc, info = env.step([tie_idx, tie_idx])
    print(f"prices={info['prices']}  quantities={info['quantities']}")
    assert np.isclose(info["quantities"][0], info["quantities"][1]), \
        "FAIL: tied prices should split demand equally"
    print("PASS: tied prices split demand 50/50\n")

    print("=== TEST 4: episode truncates after max_steps ===")
    env2 = BertrandPricingEnv(max_steps=3)
    env2.reset()
    trunc = False
    steps_taken = 0
    while not trunc:
        _, _, _, trunc, _ = env2.step([5, 5])
        steps_taken += 1
    assert steps_taken == 3, f"FAIL: expected truncation at 3 steps, got {steps_taken}"
    print(f"PASS: episode truncated after {steps_taken} steps\n")

    print("ALL TESTS PASSED")


if __name__ == "__main__":
    run_unit_tests()


=== TEST 1: both firms price at marginal cost (Bertrand NE) ===
prices=[10. 10.]  profits=[0. 0.]
PASS: both profits ~0 at p1=p2=c

=== TEST 2: firm 1 undercuts slightly below firm 2 ===
prices=[10. 30.]  quantities=[90.  0.]  profits=[0. 0.]
PASS: lower price captures full market, higher price gets zero

=== TEST 3: tie -> demand splits evenly ===
prices=[30. 30.]  quantities=[35. 35.]
PASS: tied prices split demand 50/50

=== TEST 4: episode truncates after max_steps ===
PASS: episode truncated after 3 steps

ALL TESTS PASSED


In [4]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class BertrandPricingEnv(gym.Env):
    """
    Two-firm simultaneous price-setting environment.

    Demand for firm i (linear demand Q = a - bP, split on ties):
        Q_i = a - b*p_i        if p_i <  p_j   (firm i undercuts -> full demand)
        Q_i = 0.5*(a - b*p_i)  if p_i == p_j   (tie -> split demand)
        Q_i = 0                if p_i >  p_j   (firm i priced out)

    Profit for firm i:
        pi_i = (p_i - marginal_cost) * Q_i

    Action space (per firm): Discrete(n_price_levels)
        action index -> price via a linear grid from 0 to max_price
    Observation: last round's two prices, normalized to [0, 1]
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, a=100.0, b=1.0, marginal_cost=10.0,
                 n_price_levels=25, max_price=None, max_steps=20):
        super().__init__()

        # --- configurable market parameters ---
        self.a = a                          # demand intercept
        self.b = b                          # demand slope
        self.marginal_cost = marginal_cost  # c
        self.n_price_levels = n_price_levels
        # if no ceiling given, default to the monopoly-ish price a/b so the
        # grid covers the economically meaningful range
        self.max_price = max_price if max_price is not None else (a / b)
        self.max_steps = max_steps

        # price grid: index -> actual price
        # Built in two halves so that marginal_cost is ALWAYS an exact grid
        # point, regardless of n_price_levels. This prevents the unit-test
        # failure where price_index_for(c) snaps to a nearby point and
        # produces small negative profits instead of exactly zero.
        #   lower half: n_price_levels//2 points from 0 up to (not including) c
        #   upper half: remaining points from c up to max_price (c included)
        n_lower = n_price_levels // 2
        n_upper = n_price_levels - n_lower
        lower = np.linspace(0.0, marginal_cost, n_lower, endpoint=False)
        upper = np.linspace(marginal_cost, self.max_price, n_upper)
        self.price_grid = np.concatenate([lower, upper])
        # sanity: update n_price_levels to match actual grid length
        self.n_price_levels = len(self.price_grid)

        # --- Gymnasium spaces ---
        # two firms, each picks one discrete price level
        self.action_space = spaces.MultiDiscrete([n_price_levels, n_price_levels])

        # observation: last prices for both firms, normalized to [0,1]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(2,), dtype=np.float32
        )

        self._step_count = 0
        self._last_prices = np.zeros(2, dtype=np.float32)

    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._step_count = 0
        # start both firms at price 0 (arbitrary, agents will overwrite immediately)
        self._last_prices = np.zeros(2, dtype=np.float32)
        observation = self._get_obs()
        info = {"prices": self._last_prices.copy()}
        return observation, info

    # ------------------------------------------------------------------
    def step(self, actions):
        """
        actions: array-like of two ints, each in [0, n_price_levels-1]
        """
        actions = np.asarray(actions)
        p1 = self.price_grid[actions[0]]
        p2 = self.price_grid[actions[1]]
        prices = np.array([p1, p2], dtype=np.float32)

        q1, q2 = self._demand(p1, p2)
        profit1 = (p1 - self.marginal_cost) * q1
        profit2 = (p2 - self.marginal_cost) * q2

        self._last_prices = prices
        self._step_count += 1

        observation = self._get_obs()
        reward = np.array([profit1, profit2], dtype=np.float32)

        terminated = False  # no natural "win/lose" end state
        truncated = self._step_count >= self.max_steps

        info = {
            "prices": prices,
            "quantities": np.array([q1, q2], dtype=np.float32),
            "profits": reward,
        }
        return observation, reward, terminated, truncated, info

    # ------------------------------------------------------------------
    def _demand(self, p1, p2):
        """Piecewise Bertrand demand split. Returns (q1, q2)."""
        def q_if_alone(p):
            return max(self.a - self.b * p, 0.0)

        if p1 < p2:
            return q_if_alone(p1), 0.0
        elif p2 < p1:
            return 0.0, q_if_alone(p2)
        else:  # tie -> split
            q = q_if_alone(p1)
            return 0.5 * q, 0.5 * q

    # ------------------------------------------------------------------
    def _get_obs(self):
        # normalize prices into [0,1] using max_price
        return (self._last_prices / self.max_price).astype(np.float32)

    # ------------------------------------------------------------------
    def render(self):
        p1, p2 = self._last_prices
        print(f"step={self._step_count}  p1={p1:.2f}  p2={p2:.2f}")

    # ------------------------------------------------------------------
    def price_index_for(self, price):
        """Helper: find the closest action index for a target price.
        Useful for unit testing against the analytical Nash price."""
        return int(np.argmin(np.abs(self.price_grid - price)))


# ======================================================================
# UNIT TESTS — analytical Nash price check
# ======================================================================
def run_unit_tests():
    # n_price_levels=50 gives proper resolution for the tournament AND
    # marginal_cost=10 is guaranteed to be on the grid by the constructor's
    # two-half build logic — no need to use a coarse grid as a workaround.
    env = BertrandPricingEnv(a=100, b=1, marginal_cost=10,
                              n_price_levels=50, max_price=100)
    env.reset()

    c = env.marginal_cost
    c_idx = env.price_index_for(c)

    print("=== TEST 1: both firms price at marginal cost (Bertrand NE) ===")
    obs, reward, term, trunc, info = env.step([c_idx, c_idx])
    print(f"prices={info['prices']}  profits={info['profits']}")
    assert np.allclose(info["profits"], [0, 0], atol=1.0), \
        "FAIL: profits should be ~0 when both firms price at marginal cost"
    print("PASS: both profits ~0 at p1=p2=c\n")

    print("=== TEST 2: firm 1 undercuts slightly below firm 2 ===")
    above_c_idx = env.price_index_for(c + 20)  # firm 2 prices above cost
    env.reset()
    obs, reward, term, trunc, info = env.step([c_idx, above_c_idx])
    print(f"prices={info['prices']}  quantities={info['quantities']}  profits={info['profits']}")
    assert info["quantities"][1] == 0.0, \
        "FAIL: higher-priced firm should get zero demand"
    assert info["quantities"][0] > 0.0, \
        "FAIL: lower-priced firm should capture all demand"
    print("PASS: lower price captures full market, higher price gets zero\n")

    print("=== TEST 3: tie -> demand splits evenly ===")
    tie_idx = env.price_index_for(c + 20)
    env.reset()
    obs, reward, term, trunc, info = env.step([tie_idx, tie_idx])
    print(f"prices={info['prices']}  quantities={info['quantities']}")
    assert np.isclose(info["quantities"][0], info["quantities"][1]), \
        "FAIL: tied prices should split demand equally"
    print("PASS: tied prices split demand 50/50\n")

    print("=== TEST 4: episode truncates after max_steps ===")
    env2 = BertrandPricingEnv(max_steps=3)
    env2.reset()
    trunc = False
    steps_taken = 0
    while not trunc:
        _, _, _, trunc, _ = env2.step([5, 5])
        steps_taken += 1
    assert steps_taken == 3, f"FAIL: expected truncation at 3 steps, got {steps_taken}"
    print(f"PASS: episode truncated after {steps_taken} steps\n")

    print("ALL TESTS PASSED")


if __name__ == "__main__":
    run_unit_tests()

=== TEST 1: both firms price at marginal cost (Bertrand NE) ===
prices=[10. 10.]  profits=[0. 0.]
PASS: both profits ~0 at p1=p2=c

=== TEST 2: firm 1 undercuts slightly below firm 2 ===
prices=[10.   28.75]  quantities=[90.  0.]  profits=[0. 0.]
PASS: lower price captures full market, higher price gets zero

=== TEST 3: tie -> demand splits evenly ===
prices=[28.75 28.75]  quantities=[35.625 35.625]
PASS: tied prices split demand 50/50

=== TEST 4: episode truncates after max_steps ===
PASS: episode truncated after 3 steps

ALL TESTS PASSED
